# **SpaceX  Falcon 9 First-Stage Landing Prediction**

## EDA with SQL

### Introduction
Using this Python notebook we will:

1.  Understand the Spacex DataSet
2.  Load the dataset  into the corresponding table in a Db2 database
3.  Execute SQL queries to answer assignment questions 

## Overview of the DataSet

SpaceX has gained worldwide attention for a series of historic milestones. 

It is the only private company ever to return a spacecraft from low-earth orbit, which it first accomplished in December 2010.
SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars wheras other providers cost upward of 165 million dollars each, much of the savings is because Space X can reuse the first stage. 


Therefore if we can determine if the first stage will land, we can determine the cost of a launch. 

This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

This dataset includes a record for each payload carried during a SpaceX mission into outer space.


### Download the datasets

This assignment requires you to load the spacex dataset.

In many cases the dataset to be analyzed is available as a .CSV (comma separated values) file, perhaps on the internet. Click on the link below to download and save the dataset (.CSV file):

 <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv" target="_blank">Spacex DataSet</a>



In [1]:
# Installing needed libraries

#!pip install sqlalchemy==1.3.9
#!pip install sqlalchemy==1.4.1
#Current environment has version 2.0.49

### Connect to the database

Let us first load the SQL extension and establish a connection with the database


In [2]:
# Installing additional libraries

!pip install ipython-sql
!pip install ipython-sql prettytable

In [3]:
# Loading SQL

%load_ext sql

In [4]:
# Importing libraries

import csv, sqlite3
import prettytable
import os
from pathlib import Path

prettytable.DEFAULT = 'DEFAULT'

db_path = Path.cwd().parent / "data" / "my_data1.db"

con = sqlite3.connect(db_path)
cur = con.cursor()

In [5]:
# Installing additional libraries

!pip install -q pandas

In [6]:
# Connecting to the database

db_path = Path.cwd().parent / "data" / "my_data1.db"

%sql sqlite:///{db_path.as_posix()}

In [7]:
# Getting the data from the provided location and creating a SQL table with it called 'SPACEXTBL'

import pandas as pd

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")

101

In [8]:
# Saving a local copy of the original file

file_path = Path.cwd().parent / "data" / "spacex_eda_sql.csv"
df.to_csv(file_path)

**Note:** The code below is added to remove blank rows from the table.


In [9]:
# Dropping the table if exists

%sql DROP TABLE IF EXISTS SPACEXTABLE;

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


[]

In [10]:
# Creating table 'SPACEXTABLE' only for the observations that have a date with value different than null

%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


[]

## EDA Steps

Now write and execute SQL queries to solve the assignment tasks.

**Note:** If the column names are in mixed case enclose it in double quotes
   For Example "Landing_Outcome"

### 1. Let's show the names of the unique launch sites  in the space mission


In [11]:
# Displaying the names of the unique launch sites

%sql select distinct "Launch_Site" from SPACEXTABLE;

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


### 2. Let's show 5 records where launch sites begin with the string 'CCA' 

In [12]:
# Displaying 5 records where launch site names begin with "CCA"

%sql select * from SPACEXTABLE \
     where "Launch_Site" like 'CCA%' \
     limit 5;


 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### 3. Let's show the total payload mass carried by boosters launched by NASA (CRS)


In [13]:
# Displaying the total payload mass carried by boosters launched by NASA (CRS)

%sql select sum("PAYLOAD_MASS__KG_") as 'Total Payload Mass' \
     from SPACEXTABLE \
     where "Customer" = 'NASA (CRS)';

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Total Payload Mass
45596


### 4. Let's show the average payload mass carried by booster version F9 v1.1

In [14]:
# Displaying average payload mass carried by booster version F9 v1.1

%sql select avg("PAYLOAD_MASS__KG_") as 'Average Payload Mass' \
     from SPACEXTABLE \
     where "Booster_Version" = 'F9 v1.1';

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Average Payload Mass
2928.4


### 5. List the date when the first succesful landing outcome in ground pad was acheived.

In [15]:
# Listing the date when the first succesful landing outcome in ground pad was acheived

%sql select min("Date") as 'First Successful Landing on Ground Pad' \
     from SPACEXTABLE \
     where "Landing_Outcome" = 'Success (ground pad)';

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


First Successful Landing on Ground Pad
2015-12-22


### 6. List booster names with successful drone ship landing and payload mass greater than 4000 but less than 6000

In [16]:
# Listing the names of the boosters which have success landing in drone ship and have payload mass greater than 4000 but less than 6000

%sql select distinct ("Booster_Version") \
     from SPACEXTABLE \
     where "Landing_Outcome" = 'Success (drone ship)' \
     and "PAYLOAD_MASS__KG_" > 4000 \
     and "PAYLOAD_MASS__KG_" < 6000;

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


### 7. List the total number of successful and failure mission outcomes

In [17]:
# Listing the total number of successful and failure mission outcomes

%sql select "Mission_Outcome", count(*) as 'Count' \
     from SPACEXTABLE \
     where "Mission_Outcome" like 'Success%' \
     union \
     select "Mission_Outcome", count(*) as 'Count' \
     from SPACEXTABLE \
     where "Mission_Outcome" like 'Failure%';

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Mission_Outcome,Count
Failure (in flight),1
Success,100


### 8. List all the booster_versions that have carried the maximum payload mass

In [18]:
# Listing all the booster_versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function

%sql select "Booster_Version" \
     from SPACEXTABLE \
     where "PAYLOAD_MASS__KG_" = (select max("PAYLOAD_MASS__KG_") \
                                  from SPACEXTABLE) \
     order by 1;

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1048.5
F9 B5 B1049.4
F9 B5 B1049.5
F9 B5 B1049.7
F9 B5 B1051.3
F9 B5 B1051.4
F9 B5 B1051.6
F9 B5 B1056.4
F9 B5 B1058.3


### 9. List the records with month names, failure landing_outcomes in drone ship, booster versions, launch_site in 2015

**Note:** SQLLite does not support month names. So you need to use substr(Date, 6, 2) as month to get the months and substr(Date, 1, 4) = '2015' for year.


In [19]:
# Listing the records which will display the month names, failure landing_outcomes in drone ship, booster versions, launch_site for the months in year 2015.

%sql select case substr("Date", 6, 2) \
                 when '01' then 'January' \
                 when '02' then 'February' \
                 when '03' then 'March' \
                 when '04' then 'April' \
                 when '05' then 'May' \
                 when '06' then 'June' \
                 when '07' then 'July' \
                 when '08' then 'August' \
                 when '09' then 'September' \
                 when '10' then 'October' \
                 when '11' then 'November' \
                 when '12' then 'December' \
            end as "Month", "Landing_Outcome", "Launch_Site", "Booster_Version" \
     from SPACEXTABLE \
     where substr("Date", 1, 4) = '2015' \
     and "Landing_Outcome" = 'Failure (drone ship)' \
     order by "Date", 3, 4;

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Month,Landing_Outcome,Launch_Site,Booster_Version
January,Failure (drone ship),CCAFS LC-40,F9 v1.1 B1012
April,Failure (drone ship),CCAFS LC-40,F9 v1.1 B1015


### 10. Rank the count of landing outcomes between 2010-06-04 and 2017-03-20, in descending order

In [20]:
# Ranking the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order

%sql select "Landing_Outcome", count(*) as "Count" \
     from SPACEXTABLE \
     where "Date" between '2010-06-04' and '2017-03-20' \
     group by "Landing_Outcome" \
     order by 2 desc;

 * sqlite:///E:/Data/GitHub/Data-Science-Portfolio-IBM-Professional-Certificate-Projects/10-Applied-Data-Science-Capstone/falcon9-project/data/my_data1.db
Done.


Landing_Outcome,Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1


### Reference Links

* <a href ="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Labs_Coursera_V5/labs/Lab%20-%20String%20Patterns%20-%20Sorting%20-%20Grouping/instructional-labs.md.html?origin=www.coursera.org">Hands-on Lab : String Patterns, Sorting and Grouping</a>  

*  <a  href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Labs_Coursera_V5/labs/Lab%20-%20Built-in%20functions%20/Hands-on_Lab__Built-in_Functions.md.html?origin=www.coursera.org">Hands-on Lab: Built-in functions</a>

*  <a  href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Labs_Coursera_V5/labs/Lab%20-%20Sub-queries%20and%20Nested%20SELECTs%20/instructional-labs.md.html?origin=www.coursera.org">Hands-on Lab : Sub-queries and Nested SELECT Statements</a>

*   <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Module%205/DB0201EN-Week3-1-3-SQLmagic.ipynb">Hands-on Tutorial: Accessing Databases with SQL magic</a>

*  <a href= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/Module%205/DB0201EN-Week3-1-4-Analyzing.ipynb">Hands-on Lab: Analyzing a real World Data Set</a>




---

### Original Lab Author(s)

Lakshmi Holla

### Other Contributor(s)

Rav Ahuja

<!--
## Change log
| Date | Version | Changed by | Change Description |
|------|--------|--------|---------|
| 2024-07-10 | 1.1 |Anita Verma | Changed Version|
| 2021-07-09 | 0.2 |Lakshmi Holla | Changes made in magic sql|
| 2021-05-20 | 0.1 |Lakshmi Holla | Created Initial Version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.